# v5.4.2 concurrent-centre audit

## tl;dr

- The release has **368,925 hourly rows at 239,195 UTC hours**; at most eight published events overlap at one hour.
- There are **24,840 pair-hours closer than 150 km**, but only **2 (0.01%)** contain two observed detector fixes. Almost all very-close coincidences therefore arise because at least one published centre is interpolated/posterior, not because the detector retained two co-located observations.
- `CS 1973 03`, `04`, and `05` are not one set of near-identical paths: their pairwise median separations during overlap are **633, 693, and 362 km**. The detector sees several real 850-hPa vorticity lobes, while the linker deliberately treats spatially separated lobes as distinct identities.
- NOAA IBTrACS v04r01 records two simultaneous North Indian tropical storms on 6–8 July 1973—one Bay of Bengal and one Arabian Sea—but only one Arabian Sea storm. Of the three atlas tracks, only `CS 1973 05` has even a low-confidence association with that Arabian storm (median separation 213 km); none has a publishable credible match.
- The atlas should therefore preserve the catalogue but declutter selected-track views: retain the selected centre, show only observed contemporaneous companions, and apply a synoptic-scale separation rule. Explicit date/time searches should remain the route to the fuller set.


## Context & Methods

This is a data-quality audit of the public v5.4.2 event catalogue at its intended grain: one row per physical-event UTC hour. It tests whether simultaneous published centres are observationally independent or are caused by posterior completion, then inspects the July 1973 Arabian Sea example against the frozen linker settings, ERA5 850-hPa vorticity, and local NOAA IBTrACS v04r01 input.

### Key Assumptions

- Great-circle separation is a diagnostic, not a proof of event identity.
- `position_source == "observed"` denotes an original detector-supported fix; other rows are published interpolation/posterior positions.
- IBTrACS is a useful reference for tropical storms, but it is not a complete truth set for all monsoon lows.
- The atlas fallback `CS` label is the peak **ERA5-derived IMD-style Cyclonic Storm** category. It is neither an official IMD classification nor Saffir–Simpson Category 1.


## Data

Source paths are recorded explicitly so the audit can be rerun from the shared tracking workspace.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from scipy.ndimage import maximum_filter

ATLAS_ROOT = Path.cwd()
if not (ATLAS_ROOT / "assets").is_dir():
    ATLAS_ROOT = Path.cwd().parent
WORK_ROOT = ATLAS_ROOT.parent
CATALOGUE = WORK_ROOT / "lps-v5.3-continuity-framework/tracks/lps_v5.4.2-era5-1940-2025-core-release-candidate.parquet"
IBTRACS = WORK_ROOT / "lps-v5.3-continuity-framework/inputs/ibtracs_v04r01/ibtracs.NI.list.v04r01.csv"
VORTICITY = Path("/home/users/kieran/ncas/data/era5-incompass/hourly_vorts_SA/197307.nc")

columns = [
    "track_id", "time", "lat", "lon", "position_source", "association_family_id",
    "imd_category", "imd_category_raw", "max_wind", "mean_wind",
    "pressure_deficit_hpa", "max_vort_smoothed", "candidate_quality",
    "is_mature_detection", "track_duplicate_component_size", "track_member_tracklets",
]
catalogue = pd.read_parquet(CATALOGUE, columns=columns)
catalogue["time"] = pd.to_datetime(catalogue["time"])
profile = pd.Series({
    "rows": len(catalogue),
    "tracks": catalogue["track_id"].nunique(),
    "UTC hours represented": catalogue["time"].nunique(),
    "maximum simultaneous events": catalogue.groupby("time")["track_id"].nunique().max(),
})
profile.to_frame("value")


,value
rows,368925
tracks,2980
UTC hours represented,239195
maximum simultaneous events,8


## Results

### 1. Catalogue-wide simultaneous-centre distances

In [2]:
pairs = catalogue.merge(catalogue, on="time", suffixes=("_a", "_b"))
pairs = pairs[pairs["track_id_a"] < pairs["track_id_b"]].copy()
radians = np.pi / 180
latitude_difference = (pairs["lat_b"] - pairs["lat_a"]) * radians
longitude_difference = (pairs["lon_b"] - pairs["lon_a"]) * radians
haversine = (
    np.sin(latitude_difference / 2) ** 2
    + np.cos(pairs["lat_a"] * radians)
    * np.cos(pairs["lat_b"] * radians)
    * np.sin(longitude_difference / 2) ** 2
)
pairs["separation_km"] = 6371.0088 * 2 * np.arctan2(
    np.sqrt(haversine), np.sqrt(np.maximum(0, 1 - haversine))
)
pairs["both_observed"] = (
    pairs["position_source_a"].eq("observed")
    & pairs["position_source_b"].eq("observed")
)

threshold_rows = []
for threshold in [100, 150, 180, 250, 350, 500, 600, 750, 900]:
    close = pairs[pairs["separation_km"] < threshold]
    pair_ids = close[["track_id_a", "track_id_b"]].drop_duplicates()
    threshold_rows.append({
        "threshold_km": threshold,
        "pair_hours": len(close),
        "unique_track_pairs": len(pair_ids),
        "tracks_involved": pd.unique(pair_ids.to_numpy().ravel()).size,
        "both_observed_pair_hours": int(close["both_observed"].sum()),
        "both_observed_pct": 100 * close["both_observed"].mean() if len(close) else 0,
    })
threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary


,threshold_km,pair_hours,unique_track_pairs,tracks_involved,both_observed_pair_hours,both_observed_pct
0,100,20151,958,1484,0,0.000000
1,150,24840,1031,1552,2,0.008052
2,180,27181,1072,1590,16,0.058865
3,250,31635,1157,1669,149,0.470997
4,350,37379,1386,1844,1401,3.748094
5,500,53365,1978,2199,10212,19.136138
6,600,66546,2246,2304,18727,28.141436
7,750,83490,2483,2368,30015,35.950413
8,900,96811,2675,2399,38900,40.181384


The detector's 180 km duplicate suppression is visible in the result: two independent observed fixes almost never coexist inside 150 km. Close orange points mostly represent completed paths crossing or shadowing an observed event. This supports hiding non-selected interpolated companions in a focused atlas view without deleting any catalogue rows.

### 2. July 1973 Arabian Sea tracks

In [3]:
case_ids = [3377, 3380, 3381]  # Atlas fallback labels CS 1973 03, 04, 05
case = catalogue[catalogue["track_id"].isin(case_ids)].copy()
case_summary = case.groupby("track_id").agg(
    start=("time", "min"),
    end=("time", "max"),
    hourly_positions=("time", "size"),
    observed_positions=("position_source", lambda value: int(value.eq("observed").sum())),
    interpolated_positions=("position_source", lambda value: int(value.ne("observed").sum())),
    peak_category=("imd_category", "max"),
    peak_max_wind_ms=("max_wind", "max"),
    peak_mean_wind_ms=("mean_wind", "max"),
    peak_pressure_deficit_hpa=("pressure_deficit_hpa", "max"),
    peak_smoothed_vorticity=("max_vort_smoothed", "max"),
    association_family=("association_family_id", "first"),
    duplicate_component_size=("track_duplicate_component_size", "first"),
)
case_summary


,start,end,hourly_positions,observed_positions,interpolated_positions,peak_category,peak_max_wind_ms,peak_mean_wind_ms,peak_pressure_deficit_hpa,peak_smoothed_vorticity,association_family,duplicate_component_size
track_id,,,,,,,,,,,,
3377,1973-07-04 18:00:00,1973-07-07 05:00:00,60,48,12,4.0,19.643270,10.745720,6.693298,14.097534,27476,1
3380,1973-07-05 11:00:00,1973-07-07 10:00:00,48,32,16,4.0,20.639013,14.116472,4.823486,9.007096,27467,1
3381,1973-07-06 05:00:00,1973-07-10 11:00:00,103,78,25,4.0,20.670187,15.059806,8.461853,15.503931,27477,1


In [4]:
case_pairs = pairs[
    pairs["track_id_a"].isin(case_ids) & pairs["track_id_b"].isin(case_ids)
]
case_pair_summary = case_pairs.groupby(["track_id_a", "track_id_b"]).agg(
    overlap_hours=("separation_km", "size"),
    minimum_km=("separation_km", "min"),
    median_km=("separation_km", "median"),
    maximum_km=("separation_km", "max"),
    first_overlap=("time", "min"),
    last_overlap=("time", "max"),
)
case_pair_summary.round(1)


overlap_hours  minimum_km  median_km  maximum_km  \
track_id_a track_id_b                                                     
3377       3380                   43       515.9      633.3       946.9   
           3381                   25       385.1      692.8      1144.9   
3380       3381                   30        16.8      362.5       785.0   

                            first_overlap        last_overlap  
track_id_a track_id_b                                          
3377       3380       1973-07-05 11:00:00 1973-07-07 05:00:00  
           3381       1973-07-06 05:00:00 1973-07-07 05:00:00  
3380       3381       1973-07-06 05:00:00 1973-07-07 10:00:00

All three tracks have substantial observed support, but 20–33% of their published hours are interpolated. They are too far apart for the linker's duplicate rules, whose principal overlap thresholds are 115 km median and 240 km at the 90th percentile. `CS 1973 04` and later `CS 1973 06` share liberal association-family provenance, indicating at least one fragmented family in this episode, but the published physical-event segmentation intentionally separates unsupported bridges.

### 3. Independent tropical-storm reference

In [5]:
ibtracs = pd.read_csv(IBTRACS, skiprows=[1], low_memory=False)
ibtracs["ISO_TIME"] = pd.to_datetime(ibtracs["ISO_TIME"], errors="coerce", utc=True)
official_case = ibtracs[
    ibtracs["ISO_TIME"].between("1973-07-04", "1973-07-11", inclusive="left")
][["SID", "NAME", "ISO_TIME", "LAT", "LON", "NATURE"]]
official_case.groupby("SID").agg(
    start=("ISO_TIME", "min"),
    end=("ISO_TIME", "max"),
    fixes=("ISO_TIME", "size"),
    lon_min=("LON", "min"),
    lon_max=("LON", "max"),
    lat_min=("LAT", "min"),
    lat_max=("LAT", "max"),
)


,start,end,fixes,lon_min,lon_max,lat_min,lat_max
SID,,,,,,,
1973187N19087,1973-07-06 00:00:00+00:00,1973-07-09 00:00:00+00:00,25,79.9,86.5,19.3,22.9
1973188N21071,1973-07-07 00:00:00+00:00,1973-07-08 12:00:00+00:00,13,69.3,70.6,20.5,22.3


### 4. ERA5 vorticity structure at 12 UTC on 6 July

In [6]:
with xr.open_dataset(VORTICITY) as weather:
    field = (
        weather["vo"]
        .sel(time="1973-07-06T12:00:00", level=850, latitude=slice(30, 10), longitude=slice(55, 95))
        * 1e5
    ).load()
values = field.to_numpy()
local_maxima = (
    (values == maximum_filter(values, size=(13, 13), mode="nearest"))
    & (values >= 5)
)
rows, columns = np.where(local_maxima)
vorticity_peaks = pd.DataFrame({
    "vorticity_1e-5_s-1": values[rows, columns],
    "latitude": field["latitude"].to_numpy()[rows],
    "longitude": field["longitude"].to_numpy()[columns],
}).sort_values("vorticity_1e-5_s-1", ascending=False).head(20)
vorticity_peaks.round(2)


,vorticity_1e-5_s-1,latitude,longitude
25,40.87,20.50,82.00
40,38.81,11.25,76.75
21,37.77,21.50,94.00
17,30.68,24.00,94.00
29,28.65,18.75,69.25
28,27.87,19.00,71.75
30,25.65,18.75,86.00
23,23.50,20.75,69.00
12,20.21,25.75,91.75
18,20.18,23.00,57.00


The field contains distinct maxima near 64°E, 69°E and 72°E as well as the Bay of Bengal/India features. The catalogue is therefore not inventing identical coordinates; it is promoting multiple lobes of a broad, complex circulation into separate track identities. Whether those lobes should be one meteorological event is an identity-definition problem beyond the current duplicate stage.

## Takeaways

1. **High-confidence finding:** very-close simultaneous published centres are overwhelmingly interpolation-related. Atlas companions should default to observed positions only.
2. **High-confidence finding:** the `CS` labels in this case are generated by 19.6–20.7 m s⁻¹ local maximum winds sampled inside 350 km. They do not mean Category 1 hurricane, and the UI should say so beside the selected system.
3. **Medium-confidence finding:** `CS 1973 03/04/05` represent distinct ERA5 vorticity lobes but over-partition the Arabian Sea episode relative to IBTrACS. The likely cause is the deliberately recall-first candidate/linking design plus duplicate thresholds intended only for much closer tracks.
4. **Recommended atlas remediation:** moving the hour slider must not activate weather; selected-track focus should retain the selected centre, omit interpolated companions, and collapse remaining companion centres within a synoptic-scale radius. Explicit date/time search remains available for fuller inspection.
5. **Recommended algorithm follow-up:** add a catalogue-wide audit for spatially separated but temporally overlapping tracks within the same synoptic envelope, using shared pressure/vorticity extrema and IBTrACS where applicable. Do not silently merge the frozen release without a calibrated identity experiment.
